In [2]:
import pandas as pd
import numpy as np
import joblib

df = pd.read_csv("E:/Inventory_managment/data/retail_store_inventory_cleaned.csv")
df['date'] = pd.to_datetime(df['Date'])

model = joblib.load("E:/Inventory_managment/inventory_demand_model.pkl")


In [4]:
#Prepare Latest Data for Prediction
latest_data = df.sort_values('date').groupby('Product ID').tail(1)
latest_data.head()


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,date
73044,2024-01-01,S003,P0005,Electronics,West,411,291,200,298.09,82.16,20,Sunny,1,79.32,Summer,2024-01-01
73043,2024-01-01,S003,P0004,Groceries,North,415,36,79,54.90,72.75,20,Rainy,1,72.20,Winter,2024-01-01
73042,2024-01-01,S003,P0003,Groceries,North,112,9,169,0.58,41.99,20,Snowy,0,41.69,Autumn,2024-01-01
73041,2024-01-01,S003,P0002,Groceries,South,388,229,102,234.86,39.66,0,Cloudy,0,42.94,Winter,2024-01-01
73040,2024-01-01,S003,P0001,Clothing,West,138,114,65,119.97,52.72,0,Sunny,0,52.90,Spring,2024-01-01


In [6]:
#Create Required Features

latest_data['day'] = latest_data['date'].dt.day
latest_data['month'] = latest_data['date'].dt.month
latest_data['weekday'] = latest_data['date'].dt.weekday

latest_data['lag_1'] = latest_data['Units Sold']
latest_data['lag_7'] = latest_data['Units Sold']
latest_data['rolling_mean_7'] = latest_data['Units Sold']
latest_data['rolling_mean_14'] = latest_data['Units Sold']

In [7]:
#Encode Category Column
latest_data = pd.get_dummies(latest_data, columns=['Category'], drop_first=True)


In [8]:
#Align Columns with Training Data
X_columns = model.feature_names_in_
latest_data = latest_data.reindex(columns=X_columns, fill_value=0)


In [9]:
#Predict Demand

latest_data['predicted_demand'] = model.predict(latest_data)


In [12]:
#Low-Stock Detection
latest_data['low_stock_flag'] = np.where(
    latest_data['predicted_demand'] > latest_data['Inventory Level'],
    'YES',
    'NO'
)


In [14]:
#Reorder Quantity Calculation
latest_data['reorder_quantity'] = np.where(
    latest_data['low_stock_flag'] == 'YES',
    latest_data['predicted_demand'] - latest_data['Inventory Level'],
    0
)


In [17]:
#Final Inventory Insight Table
# Ensure 'Product ID' is present (it was dropped when we reindexed to model features)
latest_data = latest_data.copy()
if 'Product ID' not in latest_data.columns:
    latest_data['Product ID'] = df.loc[latest_data.index, 'Product ID']

insight_columns = [
    'Product ID',
    'Inventory Level',
    'predicted_demand',
    'low_stock_flag',
    'reorder_quantity'
]

inventory_insights = latest_data[insight_columns]
inventory_insights.head(10)


,Product ID,Inventory Level,predicted_demand,low_stock_flag,reorder_quantity
73044,P0005,411,249.81,NO,0.0
73043,P0004,415,51.81,NO,0.0
73042,P0003,112,55.21,NO,0.0
73041,P0002,388,212.43,NO,0.0
73040,P0001,138,67.15,NO,0.0
73038,P0019,409,228.43,NO,0.0
73037,P0018,151,62.63,NO,0.0
73036,P0017,236,61.96,NO,0.0
73034,P0015,433,277.61,NO,0.0
73033,P0014,342,184.20,NO,0.0


In [18]:
#Save insight
inventory_insights.to_csv("E:/Inventory_managment/data/inventory_insights_Output.csv", index=False)